In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
#Q1 Answer 614
from datasets import load_dataset
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
dataset = dataset.map(lambda x: {'combined_text': x['prompt'] + ' ' + x['A']})
print(len(dataset['train'][51]['combined_text']))  

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


In [3]:
#Q2 Answer 30522
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print(tokenizer)
print(tokenizer.vocab_size) #30522

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
30522


In [4]:
#Q3 Answer 102
print(tokenizer.sep_token_id)

102


In [5]:
#Q4 Answer torch.Size([2000, 128])
prompts = list(dataset['train']['prompt'])
encoded = tokenizer(prompts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
print(encoded['input_ids'].shape)

#Q5 Answer Manually Hiddensize/Attentionhead 768/12 = 64 also by code below 
from transformers import BertConfig
config = BertConfig.from_pretrained('bert-base-uncased')
head_dim = config.hidden_size // config.num_attention_heads
print(head_dim)

torch.Size([2000, 128])


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

64


In [6]:
#Q6 Answer torch.Size([1, 31, 768])
from transformers import AutoModel
model = AutoModel.from_pretrained('bert-base-uncased')
prompt = dataset['train'][0]['prompt']
inputs = tokenizer(prompt, return_tensors='pt')
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [7]:
#Q7 Answer -1.2001
cls_embedding = outputs.last_hidden_state[0][0]
result = cls_embedding[:5].sum().item()
print(round(result, 4))

-1.2001


In [8]:
#Q8 Answer 0.1025
model = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
text = "Light-ion fusion is a technique."
inputs = tokenizer(text, return_tensors='pt')
print(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))
outputs = model(**inputs)
attention = outputs.attentions[-1][0][0]
fusion_idx = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]).index('fusion')
result = attention[0][fusion_idx].item()
print(round(result, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
0.1025


In [9]:
#Q9 Answer 0.7658

from sentence_transformers import SentenceTransformer, util
model_st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
prompt = dataset['train'][0]['prompt']
option_b = dataset['train'][0]['B']
emb_prompt = model_st.encode(prompt)
emb_b = model_st.encode(option_b)
score = util.cos_sim(emb_prompt, emb_b)
print(round(score.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


In [10]:
#Q10 Answer MiniLM MAP@3: 0.4231 TF-IDF miss, MiniLM hit count: 564
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

options_cols = ['A', 'B', 'C', 'D', 'E']
prompts = list(dataset['train']['prompt'])
correct_answers = list(dataset['train']['answer'])
all_options = [[dataset['train'][i][o] for o in options_cols] for i in range(len(dataset['train']))]

#MAP@3 function
def map_at_3(predictions, correct_answers):
    scores = []
    for pred, correct in zip(predictions, correct_answers):
        score = 0.0
        for k, p in enumerate(pred[:3], 1):
            if p == correct:
                score = 1.0 / k
                break
        scores.append(score)
    return np.mean(scores)

#Pipeline 1: TF-IDF
tfidf_top3 = []
for prompt, opts in zip(prompts, all_options):
    corpus = [prompt] + opts
    vec = TfidfVectorizer()
    tfidf_matrix = vec.fit_transform(corpus)
    sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])[0]
    ranked = [options_cols[i] for i in np.argsort(sims)[::-1]]
    tfidf_top3.append(ranked[:3])

#Pipeline 2: MiniLM
minilm_top3 = []
for prompt, opts in zip(prompts, all_options):
    emb_prompt = model_st.encode(prompt, convert_to_tensor=True)
    emb_opts = model_st.encode(opts, convert_to_tensor=True)
    sims = util.cos_sim(emb_prompt, emb_opts)[0]
    ranked = [options_cols[i] for i in sims.argsort(descending=True).tolist()]
    minilm_top3.append(ranked[:3])

# Results
minilm_map3 = map_at_3(minilm_top3, correct_answers)
print(f"MiniLM MAP@3: {round(minilm_map3, 4)}")

count = sum(
    1 for tfidf_pred, minilm_pred, correct in zip(tfidf_top3, minilm_top3, correct_answers)
    if correct not in tfidf_pred and correct in minilm_pred
)
print(f"TF-IDF miss, MiniLM hit count: {count}")

MiniLM MAP@3: 0.4231
TF-IDF miss, MiniLM hit count: 564


In [11]:
#Q11 Answer 0.4575
from transformers import pipeline
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
prompt = dataset['train'][1]['prompt']
candidate_labels = [dataset['train'][1]['A'], dataset['train'][1]['B'], dataset['train'][1]['C']]
result = classifier(prompt, candidate_labels)
print(round(result['scores'][0], 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.4575


In [12]:
#Q12 Answer Softmax sum: 1.0 Sigmoid sum: 0.0005 Absolute difference: 0.9995
result_multi = classifier(prompt, candidate_labels, multi_label=True)
sum_softmax = sum(result['scores'])
sum_sigmoid = sum(result_multi['scores'])
print(f"Softmax sum: {round(sum_softmax, 4)}")
print(f"Sigmoid sum: {round(sum_sigmoid, 4)}")
print(f"Absolute difference: {round(abs(sum_softmax - sum_sigmoid), 4)}")

Softmax sum: 1.0
Sigmoid sum: 0.0005
Absolute difference: 0.9995


In [13]:
#Q13 Answer B In this we have directly used the model beacause text2textgeneration is nowhere available
from transformers import T5ForConditionalGeneration, T5Tokenizer
tokenizer_t5 = T5Tokenizer.from_pretrained('google/flan-t5-small')
model_t5 = T5ForConditionalGeneration.from_pretrained('google/flan-t5-small')
prompt = dataset['train'][0]['prompt']
a = dataset['train'][0]['A']
b = dataset['train'][0]['B']
input_text = f"Question: {prompt}. Is the correct answer A: {a} or B: {b}? Answer with just the letter A or B."
input_ids = tokenizer_t5(input_text, return_tensors='pt').input_ids
outputs = model_t5.generate(input_ids, max_new_tokens=5)
print(tokenizer_t5.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


B
